In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
df = pd.read_hdf("pems-bay.h5")

print(df.shape)

df.head()

(52116, 325)


sensor_id,400001,400017,400030,400040,400045,400052,400057,400059,400065,400069,...,409525,409526,409528,409529,413026,413845,413877,413878,414284,414694
2017-01-01 00:00:00,71.4,67.8,70.5,67.4,68.8,66.6,66.8,68.0,66.8,69.0,...,68.8,67.9,68.8,68.0,69.2,68.9,70.4,68.8,71.1,68.0
2017-01-01 00:05:00,71.6,67.5,70.6,67.5,68.7,66.6,66.8,67.8,66.5,68.2,...,68.4,67.3,68.4,67.6,70.4,68.8,70.1,68.4,70.8,67.4
2017-01-01 00:10:00,71.6,67.6,70.2,67.4,68.7,66.1,66.8,67.8,66.2,67.8,...,68.4,67.4,68.4,67.5,70.2,68.3,69.8,68.4,70.5,67.9
2017-01-01 00:15:00,71.1,67.5,70.3,68.0,68.5,66.7,66.6,67.7,65.9,67.8,...,68.5,67.5,68.5,67.5,70.4,68.7,70.2,68.4,70.8,67.6
2017-01-01 00:20:00,71.7,67.8,70.2,68.1,68.4,66.9,66.1,67.7,66.1,67.8,...,68.5,67.7,68.5,67.4,69.6,69.1,70.0,68.4,71.0,67.9


In [3]:
scaler = MinMaxScaler()

data_scaled = scaler.fit_transform(
    df.values
)

print(data_scaled.shape)

(52116, 325)


In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(data_scaled) - sequence_length
):

    X.append(
        data_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        data_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(52104, 12, 325)
(52104, 325)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(41683, 12, 325)
(10421, 12, 325)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

print(X_train.shape)

torch.Size([41683, 12, 325])


In [7]:
import pickle
import numpy as np

with open("adj_mx_bay.pkl", "rb") as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(
        f,
        encoding="latin1"
    )

print(adj_mx.shape)

print(adj_mx.min())
print(adj_mx.max())

(325, 325)
0.0
1.0


In [8]:
A = adj_mx

A = A + np.eye(A.shape[0])

D = np.diag(
    np.power(
        A.sum(axis=1),
        -0.5
    )
)

A_hat = D @ A @ D

print(A_hat.shape)

(325, 325)


In [9]:
import torch

A_hat = torch.tensor(
    A_hat,
    dtype=torch.float32
)

print(A_hat.shape)

torch.Size([325, 325])


In [10]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [11]:
import torch.nn as nn

class TemporalConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(3,1),
            padding=(1,0)
        )

    def forward(self,x):

        return torch.relu(
            self.conv(x)
        )

In [12]:
class AttentionGraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        channels
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.attention = nn.Linear(
            16,
            16
        )

        self.weight = nn.Linear(
            channels,
            channels
        )

    def forward(self,x):

        E = self.attention(
            self.node_embeddings
        )

        adj = torch.softmax(
            torch.relu(
                E @ E.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,bctj->bcti",
            adj,
            x
        )

        x = x.permute(
            0,
            2,
            3,
            1
        )

        x = self.weight(x)

        x = x.permute(
            0,
            3,
            1,
            2
        )

        return torch.relu(x)

In [13]:
class AGCSTGCN(nn.Module):

    def __init__(self):

        super().__init__()

        self.temp1 = TemporalConv(
            1,
            32
        )

        self.graph = AttentionGraphConv(
            num_nodes=325,
            channels=32
        )

        self.temp2 = TemporalConv(
            32,
            32
        )

        self.fc = nn.Linear(
            32,
            1
        )

    def forward(self,x):

        x = x.unsqueeze(1)

        x = self.temp1(x)

        x = self.graph(x)

        x = self.temp2(x)

        x = x.mean(dim=2)

        x = x.permute(
            0,
            2,
            1
        )

        x = self.fc(x)

        return x.squeeze(-1)

In [14]:
model = AGCSTGCN()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 325])
torch.Size([64, 325])


In [15]:
model = AGCSTGCN()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.009605
Epoch 2/30 Loss: 0.001150
Epoch 3/30 Loss: 0.000835
Epoch 4/30 Loss: 0.000734
Epoch 5/30 Loss: 0.000668
Epoch 6/30 Loss: 0.000643
Epoch 7/30 Loss: 0.000613
Epoch 8/30 Loss: 0.000604
Epoch 9/30 Loss: 0.000594
Epoch 10/30 Loss: 0.000580
Epoch 11/30 Loss: 0.000572
Epoch 12/30 Loss: 0.000577
Epoch 13/30 Loss: 0.000572
Epoch 14/30 Loss: 0.000550
Epoch 15/30 Loss: 0.000544
Epoch 16/30 Loss: 0.000541
Epoch 17/30 Loss: 0.000535
Epoch 18/30 Loss: 0.000537
Epoch 19/30 Loss: 0.000529
Epoch 20/30 Loss: 0.000523
Epoch 21/30 Loss: 0.000520
Epoch 22/30 Loss: 0.000518
Epoch 23/30 Loss: 0.000512
Epoch 24/30 Loss: 0.000510
Epoch 25/30 Loss: 0.000511
Epoch 26/30 Loss: 0.000506
Epoch 27/30 Loss: 0.000509
Epoch 28/30 Loss: 0.000506
Epoch 29/30 Loss: 0.000508
Epoch 30/30 Loss: 0.000503


In [16]:
train_time = time.time() - train_start
print("Time Taken:", train_time)

Time Taken: 5529.490921258926


In [17]:
torch.save(
    model.state_dict(),
    "AGC-STGCN-PEMS-BAY.pth"
)

In [18]:
import psutil

print(
    psutil.virtual_memory()
)

svmem(total=16833396736, available=3508195328, percent=79.2, used=13325201408, free=3508195328)


In [19]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()
infer_start = time.time()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.cpu().numpy()
        )

        all_targets.append(
            y_batch.cpu().numpy()
        )

predictions = np.concatenate(
    all_predictions,
    axis=0
)

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 15.47527027130127
MAE: 0.012919546
RMSE: 0.023583349


In [20]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)

print("MAPE:", mape)
print("R2:", r2)

MAPE: 5704.377365112305
R2: 0.9703495552462618


In [21]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 9793
